# Tầng 2: Qwen gắn cờ nghĩa cho đuôi dài của description (Colab T4)

Notebook này chạy **trên Google Colab**, không chạy trên máy. Nó đọc `P10_tang2_cho_qwen.csv` (130 nghìn cặp zh–vi chưa ai kiểm nghĩa) và gắn cho mỗi cặp một nhãn:
- `DUNG`: bản Việt đúng nghĩa;
- `SAI_NGHIA`: sai nghĩa (sai nghĩa từ, sai số/đơn vị, sai loại hàng, thiếu/thừa thông tin quan trọng);
- `KHONG_CHAC`: không chắc.

Qwen **chỉ gắn cờ**. Notebook pipeline (`desc-pipeline.ipynb`) đo độ chính xác của cờ `SAI_NGHIA` trên 300 cặp Claude đã chấm trước (`P10_qwen_hieu_chinh.csv`); chỉ khi độ chính xác ≥ 80% (và ≥ 20 cờ) thì cặp bị cờ mới bị loại khỏi train.

**Mô hình:** `Qwen/Qwen2.5-7B-Instruct` (Apache 2.0), nạp 4-bit bằng bitsandbytes (bản float16 đầy đủ ~15 GB không vừa T4). Không sinh chữ: đọc xác suất 3 đáp án A/B/C ở token đầu tiên, nên nhanh và cho kết quả cố định.

## Cách chạy
1. Mở notebook **trên trình duyệt** (colab.research.google.com; extension Colab trong VS Code thường không mount được Drive). **Runtime → Change runtime type → T4 GPU**.
2. Trên Google Drive tạo thư mục `MyDrive/zh-vi-desc-qwen/`, tải file `data/processed/desc/duyet/P10_tang2_cho_qwen.csv` của máy lên đó. Khi ô kết nối Drive hiện hộp thoại: Connect → chọn tài khoản → **tick đủ các quyền** → Continue. Nếu vẫn không kết nối được, notebook tự chuyển sang chế độ không Drive và hỏi tải file lên (khi đó kết quả nằm trong phiên Colab, mất nếu bị ngắt).
3. Tải thêm `data/processed/desc/duyet/P10_qwen_hieu_chinh.csv` vào cùng thư mục. Chạy lần lượt các ô. Ô **chỉnh ngưỡng** chấm 100 cặp hiệu chỉnh đợt 1, in bảng độ chính xác theo ngưỡng và ước tính tổng thời gian; **gửi Claude bảng này** rồi mới chạy ô **chạy hết**.
4. Kết quả (xác suất) ghi từng lô 1.000 cặp vào `MyDrive/zh-vi-desc-qwen/lo_v2/`. Mất kết nối thì mở lại, chạy lại từ đầu: lô đã xong sẽ được bỏ qua.
5. Ô cuối gắn nhãn theo ngưỡng đã chọn và gộp thành `P10_tang2_ket_qua.csv` (nếu mở phiên mới, chạy lại ô chỉnh ngưỡng trước). Tải file này về, chép vào `data/processed/desc/duyet/` trên máy, rồi chạy lại `desc-pipeline.ipynb`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Thư viện: transformers có sẵn trên Colab; cần thêm bitsandbytes để nạp 4-bit
!pip -q install -U bitsandbytes accelerate

In [ ]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from google.colab import drive, files

# Kết nối Drive để lưu từng lô (chạy tiếp được khi mất kết nối). Khi hộp thoại hiện ra: Connect → chọn tài khoản →
# tick ĐỦ các quyền → Continue. Nếu vẫn lỗi (popup bị chặn, hoặc chạy qua extension VS Code) thì chuyển sang chế độ
# không Drive: tải file lên phiên Colab, kết quả lưu trong /content (MẤT nếu phiên bị ngắt, nhớ tải về kịp).
DUNG_DRIVE = True
try:
    drive.mount("/content/drive", force_remount=True, timeout_ms=300000)
except Exception as e:
    DUNG_DRIVE = False
    print("Không kết nối được Drive:", e, "\n→ chuyển sang chế độ không Drive")

THU_MUC = Path("/content/drive/MyDrive/zh-vi-desc-qwen") if DUNG_DRIVE else Path("/content/zh-vi-desc-qwen")
THU_MUC.mkdir(parents=True, exist_ok=True)
VAO = THU_MUC / "P10_tang2_cho_qwen.csv"
LO = THU_MUC / "lo_v2"   # v2: câu hỏi ngắn + ngưỡng (03/10); không trộn với kết quả câu hỏi cũ
LO.mkdir(parents=True, exist_ok=True)
if not VAO.exists():
    print(f"Chưa thấy {VAO}. Chọn file P10_tang2_cho_qwen.csv trên máy để tải lên:")
    up = files.upload()
    ten = next(iter(up))
    VAO.write_bytes(up[ten])
assert VAO.exists()
print("Thư mục làm việc:", THU_MUC, "| dùng Drive:", DUNG_DRIVE)

df = pd.read_csv(VAO, dtype=str, keep_default_na=False, encoding="utf-8-sig")
print(f"{len(df):,} cặp; cột: {list(df.columns)}")
# Sắp theo độ dài để mỗi lô có độ dài gần nhau (đỡ phần đệm, chạy nhanh hơn); giữ chỉ số gốc
df["_dai"] = (df.gia_tri_zh.str.len() + df.gia_tri_vi.str.len()).clip(upper=400)
df = df.sort_values("_dai", kind="stable").reset_index(drop=True)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

import gc

MODEL = "Qwen/Qwen2.5-7B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
# Chạy lại ô này trong cùng phiên thì KHÔNG nạp bản thứ hai (từng gây CUDA out of memory 03/10)
if "model" in globals():
    print("Mô hình đã nạp trong phiên này: dùng lại, không nạp thêm")
else:
    gc.collect(); torch.cuda.empty_cache()
    model = AutoModelForCausalLM.from_pretrained(
        MODEL,
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                               bnb_4bit_compute_dtype=torch.float16),
        device_map="cuda:0", torch_dtype=torch.float16)
    model.eval()
ID = {k: tok.encode(k, add_special_tokens=False)[0] for k in "ABC"}
print("token đáp án:", ID, "| GPU dùng:", round(torch.cuda.memory_allocated() / 2**30, 1), "GB")

## Câu hỏi gửi Qwen

Câu hỏi bằng tiếng Anh (Qwen theo dõi chỉ dẫn tiếng Anh tốt nhất), nội dung giữ nguyên tiếng Trung và tiếng Việt. Theo quyết định của nhóm, **tiếng Anh/pinyin còn sót trong bản Việt không tính là lỗi** (phần đó Quang xử lý sau). Các ví dụ mẫu lấy từ từ điển đã duyệt, không nằm trong 300 cặp hiệu chỉnh.

In [ ]:
# Câu hỏi ngắn (~80 token): bản dài 03/10 chạy 8,3 giờ và gắn cờ 56% (quá tay)
HUONG_DAN = ("Check a Chinese to Vietnamese e-commerce translation. Reply A if the meaning is correct, "
             "B if there is a meaning error (wrong word sense, wrong number or unit, wrong product type, key information "
             "missing or added), C if unsure. Ignore untranslated English or pinyin, capitalization, punctuation and style.")


def cau_hoi(ten, zh, vi):
    msgs = [{"role": "system", "content": HUONG_DAN},
            {"role": "user", "content": f"Attribute: {ten}\nZH: {zh}\nVI: {vi}"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def cham(khung, co_lo=32):
    ra = []
    for i in range(0, len(khung), co_lo):
        b = khung.iloc[i:i + co_lo]
        enc = tok([cau_hoi(t, z, v) for t, z, v in zip(b.ten_thuoc_tinh, b.gia_tri_zh, b.gia_tri_vi)],
                  return_tensors="pt", padding=True, truncation=True, max_length=640).to("cuda:0")
        logit = model(**enc).logits[:, -1, :]
        p = torch.softmax(logit[:, [ID["A"], ID["B"], ID["C"]]].float(), dim=-1).cpu().numpy()
        ra.append(p)
    p = np.vstack(ra)
    return khung.assign(p_dung=p[:, 0].round(5), p_sai=p[:, 1].round(5), p_kc=p[:, 2].round(5))


NGUONG = None   # ngưỡng p_sai để gắn SAI_NGHIA; chọn ở ô chỉnh ngưỡng bên dưới


def gan_nhan(kq, nguong):
    return kq.assign(nhan_qwen=np.where(kq.p_sai >= nguong, "SAI_NGHIA", "DUNG"))

## Chỉnh ngưỡng trên 100 cặp hiệu chỉnh đợt 1 (và đo tốc độ)

Cần thêm file `P10_qwen_hieu_chinh.csv` (trong `data/processed/desc/duyet/` trên máy) ở cùng thư mục làm việc. Ô này **chỉ dùng 100 cặp đợt 1** (`dot = mau100_02-10`) để chọn ngưỡng; 200 cặp đợt 2 được giữ riêng, notebook pipeline dùng chúng để đo độ chính xác thật (không bị "học tủ").

Ngưỡng được chọn: ngưỡng nhỏ nhất mà độ chính xác của cờ ≥ 85% trên 100 cặp (để dư so với yêu cầu 80%). Gửi Claude bảng in ra trước khi chạy hết.

In [ ]:
HC = THU_MUC / "P10_qwen_hieu_chinh.csv"
if not HC.exists():
    print("Chọn file P10_qwen_hieu_chinh.csv trên máy để tải lên:")
    up = files.upload()
    HC.write_bytes(up[next(iter(up))])
hc = pd.read_csv(HC, dtype=str, keep_default_na=False, encoding="utf-8-sig")
chinh = hc[hc["dot"] == "mau100_02-10"].copy()
t0 = time.time()
kq_c = cham(chinh)
giay = time.time() - t0
print(f"{len(chinh)} cặp: {giay:.0f} giây → ước tính {len(df) * giay / len(chinh) / 3600:.1f} giờ cho {len(df):,} cặp")
sai = kq_c.claude_nhan.eq("SAI")
bang = []
for t in [0.5, 0.8, 0.9, 0.95, 0.98, 0.99, 0.995, 0.999]:
    co = kq_c.p_sai >= t
    bang.append({"nguong": t, "so_co": int(co.sum()), "precision": round(sai[co].mean(), 3) if co.any() else None,
                 "recall": round(co[sai].mean(), 3)})
bang = pd.DataFrame(bang)
display(bang)
dat = bang[(bang.precision >= 0.85) & (bang.so_co >= 5)]
NGUONG = float(dat.nguong.min()) if len(dat) else None
print("Ngưỡng chọn:", NGUONG, "(None = không ngưỡng nào đạt: dừng, báo Claude)")
pd.set_option("display.max_colwidth", 60)
display(kq_c.sort_values("p_sai", ascending=False)[["gia_tri_zh", "gia_tri_vi", "claude_nhan", "p_sai"]].head(25))

## Chạy hết (theo lô 1.000 cặp, chạy tiếp được khi mất kết nối)

In [ ]:
assert NGUONG is not None, "Chưa có ngưỡng đạt yêu cầu: dừng lại, gửi bảng ở ô trên cho Claude"
CO_LO_FILE = 1000
so_lo = (len(df) + CO_LO_FILE - 1) // CO_LO_FILE
t0 = time.time()
for j in range(so_lo):
    f = LO / f"lo_{j:05d}.csv"
    if f.exists():
        continue
    kq = cham(df.iloc[j * CO_LO_FILE:(j + 1) * CO_LO_FILE])
    kq.drop(columns="_dai").to_csv(f, index=False, encoding="utf-8-sig")
    xong = len(list(LO.glob("lo_*.csv")))
    print(f"lô {j + 1}/{so_lo} xong; đã có {xong}/{so_lo} lô; {(time.time() - t0) / 60:.0f} phút trong phiên này")
print("Xong tất cả các lô.")

## Gộp kết quả

Ghi `P10_tang2_ket_qua.csv` vào thư mục Drive. Tải về và chép vào `data/processed/desc/duyet/` trên máy.

In [ ]:
lo = sorted(LO.glob("lo_*.csv"))
assert len(lo) == so_lo, f"mới có {len(lo)}/{so_lo} lô: chạy lại ô trên cho xong"
kq = pd.concat([pd.read_csv(f, dtype=str, keep_default_na=False, encoding="utf-8-sig") for f in lo], ignore_index=True)
assert len(kq) == len(df) and kq[["gia_tri_zh", "gia_tri_vi"]].duplicated().sum() == 0
assert NGUONG is not None, "chạy lại ô chỉnh ngưỡng trước (phiên mới thì biến NGUONG mất)"
kq = gan_nhan(kq.assign(p_sai=kq.p_sai.astype(float)), NGUONG)
print("Ngưỡng p_sai:", NGUONG)
cot =["gia_tri_zh", "gia_tri_vi", "ten_thuoc_tinh", "so_lan", "nhan_qwen", "p_dung", "p_sai", "p_kc"]
kq[cot].to_csv(THU_MUC / "P10_tang2_ket_qua.csv", index=False, encoding="utf-8-sig")
print(kq.nhan_qwen.value_counts(), "\nĐã ghi", THU_MUC / "P10_tang2_ket_qua.csv")
if not DUNG_DRIVE:   # chế độ không Drive: tải thẳng về máy
    files.download(str(THU_MUC / "P10_tang2_ket_qua.csv"))